In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('mental.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.describe()

In [145]:
df['Most_Used_Platform'].unique()

<StringArray>
[ 'Facebook',  'LinkedIn', 'Instagram',  'Snapchat',   'Twitter',   'YouTube',
    'TikTok',      'LINE', 'KakaoTalk', 'VKontakte',  'WhatsApp',    'WeChat']
Length: 12, dtype: str

EDA

In [ ]:
sns.histplot(df['Mental_Health_Score'], kde=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
order = [ 'Low','Medium',  'High','Very High']
sns.boxplot(x='Stress_Level', y='Mental_Health_Score', data=df, order=order)

In [ ]:
sns.scatterplot(x='Study_Hours', y='Mental_Health_Score', data=df)

In [ ]:
sns.scatterplot(x='Physical_Activity_Hours', y='Mental_Health_Score', data=df)

In [ ]:
sns.scatterplot(x='Avg_Daily_Usage_Hours', y='Mental_Health_Score', data=df)

In [ ]:
sns.scatterplot(x='Sleep_Hours_Per_Night', y='Mental_Health_Score', data=df)

In [ ]:
sns.countplot(x=df['Most_Used_Platform'])
plt.xticks(rotation=90)

CHECKING OUTLIERS

In [ ]:
num_features = df.select_dtypes(include='number')

q1 = num_features.quantile(0.25)
q3 = num_features.quantile(0.75)

IQR = q3 - q1

lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR

outlier_mask = (
    (num_features < lower_bound) |
    (num_features > upper_bound)
).any(axis=1)

df = df.loc[~outlier_mask].copy()

DATA CLEANING

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
num_features.skew()

FEATURE ENGG

In [143]:
top_countries = df['Country'].value_counts().head(11).index

def group_country(country):
    if country in top_countries:
        return country
    else:
        return 'Other'

df['Grouped_Country'] = df["Country"].apply(group_country)

df['Grouped_Country'].unique()

<StringArray>
[    'Other',    'Canada',       'USA',     'India', 'Australia',        'UK',
   'Germany',    'France',     'Spain',    'Mexico',    'Turkey']
Length: 11, dtype: str

In [ ]:
df.head()

TRAIN TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

skewed_col = ['Study_Hours']
other_num_col = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ordinal_col = ['Stress_Level']
normal_col = ['Gender','Academic_Level','Most_Used_Platform','Purpose_Of_Use','Grouped_Country']

feature_col = skewed_col + other_num_col + ordinal_col + normal_col

X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ENCODING PIPELINE CREATION

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

skewed_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

plain_numeric_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))
])

normal_pipeline = Pipeline(steps=[
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('Skewed_Pipeline',skewed_pipeline, skewed_col),
    ('Numeric_Pipeline',plain_numeric_pipeline, other_num_col),
    ('Ordinal_Pipeline',ordinal_pipeline, ordinal_col),
    ('Normal_Pipeline',normal_pipeline, normal_col)
])

MODEL TRAINING

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('linear_regression', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)
lr_pred_train = lr_pipeline.predict(X_train)

print('Testing Accuracy ->',round(r2_score(y_train, lr_pred_train) * 100, 2))
print('Training Accuracy ->', round(r2_score(y_test, lr_pred) * 100, 2))
print('MAE ->', mean_absolute_error(y_test, lr_pred))
print('MSE ->',mean_squared_error(y_test, lr_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_pred_train = rf_pipeline.predict(X_train)

print('Training Accuracy ->', round(r2_score(y_test, rf_pred) * 100, 2))
print('Testing Accuracy ->',round(r2_score(y_train, rf_pred_train) * 100, 2))
print('MAE ->', mean_absolute_error(y_test, rf_pred))
print('MSE ->',mean_squared_error(y_test, rf_pred))

USING RANDOM CROSS VALIDATION

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    'random_forest__n_estimators': [200, 300, 500],
    'random_forest__max_depth': [5, 8, 10, 15],
    'random_forest__min_samples_split': [10, 20, 30, 50],
    'random_forest__min_samples_leaf': [5, 10, 15, 20],
    'random_forest__max_features': [0.5, 0.7, 0.9]
}

random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=params,
    n_iter=15,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random.fit(X_train,y_train)

rn_pred = random.predict(X_test)
rn_pred_training = random.predict(X_train)

print("Best Parameters:", random.best_params_)
print("Best CV R2:", round(random.best_score_ * 100, 2), "%")

print('test ->',round(r2_score(y_test,rn_pred),2))
print('train ->',round(r2_score(y_train,rn_pred_training),2))

EXPORTING MODEL USING JOB_LIB

In [ ]:
import joblib 

joblib.dump(rf_pipeline, 'Mental_Model.pkl')